# 04 — Conversion Analysis

This notebook measures purchase conversion and compares rates across the segments available in the Retailrocket-derived dataset.

## Coverage
- Conversion outcome definition
- Cohort and observation windows
- Segment conversion rates
- Confidence intervals
- Lift and statistical comparisons
- Export of conversion analysis tables

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run the real-data pipeline first")

df = pd.read_csv(DATA_PATH)
product_users = set(df.loc[df["stage"] == "product_view", "user_id"])
purchase_users = set(df.loc[df["stage"] == "purchase", "user_id"])
rate = len(product_users & purchase_users) / len(product_users) if product_users else 0.0
print({"product_view_users": len(product_users), "purchasers": len(purchase_users), "view_to_purchase_rate": rate})

rows = []
for column in ["campaign", "channel", "device"]:
    if column not in df:
        continue
    for segment, frame in df.groupby(column):
        views = set(frame.loc[frame["stage"] == "product_view", "user_id"])
        purchases = set(frame.loc[frame["stage"] == "purchase", "user_id"])
        rows.append({"dimension": column, "segment": str(segment), "view_users": len(views), "purchasers": len(purchases), "conversion_rate": len(views & purchases) / len(views) if views else 0.0})

segment_metrics = pd.DataFrame(rows)
display(segment_metrics.sort_values(["dimension", "conversion_rate"], ascending=[True, False]))


{'product_view_users': 1404179, 'purchasers': 11719, 'view_to_purchase_rate': 0.008040997622098037}


,dimension,segment,view_users,purchasers,conversion_rate
0,campaign,unknown,1404179,11719,0.008041
1,channel,unknown,1404179,11719,0.008041
2,device,unknown,1404179,11719,0.008041
